# Preparação do ambiente

In [1]:
%pip install pandas
%pip install openpyxl
%pip install plotly
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import glob
import os

# Funções

In [3]:
def label_semana(df):
    s = df["Data"]           # sua coluna de datas
    mask = s.notna()

    m_ini = s.dt.to_period("M").dt.start_time
    m_fim = (m_ini + pd.offsets.MonthEnd(0))

    # 1) segunda e domingo da semana real (Seg–Dom)
    week_mon = s - pd.to_timedelta(s.dt.weekday, unit="D")       # segunda
    week_sun = week_mon + pd.Timedelta(days=6)                   # domingo

    # 2) ajusta para ficar dentro do mês
    week_start = week_mon.where(week_mon >= m_ini, m_ini)        # início = max(seg, 1º dia do mês)
    week_end   = week_sun.where(week_sun <= m_fim, m_fim)        # fim    = min(dom, último dia do mês)

    # 3) rótulo
    mes_abrev = {1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",
                7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"}

    w0 = m_ini.dt.weekday
    week_in_month = ((s.dt.day + w0 - 1) // 7 + 1).astype("Int64")

    label = pd.Series(pd.NA, index=s.index, dtype="string")
    label[mask] = (
        s[mask].dt.year.astype(str)
        + " - "
        + s[mask].dt.month.astype(str)
        + " "
        + s[mask].dt.month.map(mes_abrev)
        + " - Sem " + week_in_month[mask].astype(str)
        + " - " + week_start[mask].dt.day.astype(str).str.zfill(2)
        + " a " + week_end[mask].dt.day.astype(str).str.zfill(2)
    )

    df["Semana_mes"] = week_in_month
    df["Semana_label"] = label
    return df

In [37]:
def grafico_barra_linha(df, col_x):

    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # garante ordem cronológica
    df_plot = df.copy()

    df_plot["Label"] = df_plot[col_x].copy()

    # cria figura com eixo secundário
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # ==== BARRAS ====
    fig.add_trace(
        go.Bar(
            x=df_plot["Label"],
            y=df_plot["Oferta"],
            name="Oferta",
            text=df_plot["Oferta"],
            textposition="outside"
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Bar(
            x=df_plot["Label"],
            y=df_plot["Ocupação"],
            name="Ocupação",
            text=df_plot["Ocupação"],
            textposition="outside"
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Bar(
            x=df_plot["Label"],
            y=df_plot["Realizado"],
            name="Realizado",
            text=df_plot["Realizado"],
            textposition="outside"
        ),
        secondary_y=False
    )

    # ==== LINHAS (% ) ====
    fig.add_trace(
        go.Scatter(
            x=df_plot["Label"],
            y=df_plot["% Ocupação"],
            name="% Ocupação",
            mode="lines+markers+text",
            text=df_plot["% Ocupação"].round(0).astype(int).astype(str) + "%",
            textposition="top center"
        ),
        secondary_y=True
    )

    fig.add_trace(
        go.Scatter(
            x=df_plot["Label"],
            y=df_plot["% Realizado"],
            name="% Realizado",
            mode="lines+markers+text",
            text=df_plot["% Realizado"].round(0).astype(int).astype(str) + "%",
            textposition="bottom center"
        ),
        secondary_y=True
    )

    # ==== LAYOUT ====
    fig.update_layout(
        title="Oferta, Ocupação e Realizado por Mês",
        barmode="group",              # barras lado a lado
        xaxis_title=col_x,
        legend_title="Indicadores",
        hovermode="x unified",
        margin=dict(t=80, l=40, r=40, b=80)
    )

    # eixo esquerdo: quantidades
    fig.update_yaxes(
        title_text="Agendas",
        secondary_y=False
    )

    # eixo direito: percentuais 0–100%
    fig.update_yaxes(
        title_text="Percentual",
        range=[0, 100],
        secondary_y=True
    )

    return fig.show()

In [5]:
def tratar_nomes_nutri(df, coluna='Nutri'):
    """
    Padroniza nomes de nutricionistas corrigindo caracteres corrompidos.
    """
    mapa_nomes = {
        'ANA LU SA R  SZAJUBOK': 'ANA LUÍSA R. SZAJUBOK',
        'C NTIA DOS SANTOS IRINEU': 'CÍNTIA DOS SANTOS IRINEU',
        'CINTIA DOS SANTOS IRINEU': 'CÍNTIA DOS SANTOS IRINEU',
    }
    df[coluna] = df[coluna].str.upper().replace(mapa_nomes)
    return df

# Dicionários

In [6]:
dict_dia_semana = {
    0: "Seg",
    1: "Ter",
    2: "Qua",
    3: "Qui",
    4: "Sex",
    5: "Sáb",
    6: "Dom",
}

In [7]:
dict_mes = {1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",
                7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"}

# ETL

## Input A - Análise de Disponibilidade

In [8]:
df_input_a = pd.read_excel(r'C:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\arquivos originais\Arquivos fechamento Jan_2026\01 Disponibilidade Optum - cópia bruta do sistema.xlsx')
df_input_a.head()

,Dia,HORA INICIAL,HORA FINAL,HORAS TOTAIS,TIPO ATENDIMENTO,RESPONSÁVEL,Unnamed: 6
0,NaN,05/01/2026 - Seg,07:00:00,09:00:00,02:00:00,NaN,DANIELLE CASTELLANI
1,NaN,05/01/2026 - Seg,09:00:00,12:00:00,03:00:00,NaN,MANOELA MARIOTTO LANZARA
2,NaN,05/01/2026 - Seg,11:00:00,12:00:00,01:00:00,NaN,JULIANA RIBEIRO DE MELO
3,NaN,05/01/2026 - Seg,12:00:00,13:00:00,01:00:00,NaN,JULIANA RIBEIRO DE MELO
4,NaN,05/01/2026 - Seg,13:00:00,17:00:00,04:00:00,NaN,ANDREIA DUTRA GONCALVES


In [9]:
dict_mes = {1:'Janeiro', 2: 'Fevereiro', 3:'Março', 4:'Abril', 5:'Maio', 6:'Junho',
            7:'Julho', 8:'Agosto', 9:'Setembro', 10:'Outubro', 11:'Novembro', 12:'Dezembro'}

df_input_a_trd = pd.DataFrame(columns=['Data completa', 'Ano', 'Mês', 'Mês_Ano', 'DDS', 'Início', 'Fim', 'Total horas', 'Janelas','Nutri'])
df_input_a_trd['Data completa'] = df_input_a['HORA INICIAL'].copy()
df_input_a_trd["Data"] = df_input_a_trd["Data completa"].str.split(" -").str[0]
df_input_a_trd["Data"] = pd.to_datetime(df_input_a_trd["Data"], format="%d/%m/%Y", errors="coerce")
df_input_a_trd['Ano'] = df_input_a_trd["Data"].dt.year
df_input_a_trd['Mês_num'] = df_input_a_trd["Data"].dt.month
df_input_a_trd['Mês'] = df_input_a_trd['Mês_num'].map(dict_mes)
df_input_a_trd['Mês_Ano'] = df_input_a_trd['Mês'] + df_input_a_trd['Ano'].astype(str)
df_input_a_trd['DDS'] = df_input_a_trd['Data completa'].str[-3:]
df_input_a_trd['Início'] = df_input_a['HORA FINAL'].copy()
df_input_a_trd["Início"] = pd.to_datetime(df_input_a_trd["Início"], format="%H:%M:%S", errors="coerce")
df_input_a_trd['Fim'] = df_input_a['HORAS TOTAIS'].copy()
df_input_a_trd["Fim"]    = pd.to_datetime(df_input_a_trd["Fim"], format="%H:%M:%S", errors="coerce")
df_input_a_trd["Total horas"] = (df_input_a_trd["Fim"] - df_input_a_trd["Início"])
df_input_a_trd["Total horas"] = df_input_a_trd["Total horas"].apply(lambda x: str(x).split(" days ")[-1] if pd.notnull(x) else None)
df_input_a_trd["Janelas"] = (df_input_a_trd["Fim"] - df_input_a_trd["Início"]).dt.total_seconds() / 3600
df_input_a_trd["Janelas"] = df_input_a_trd["Janelas"].astype(int)
df_input_a_trd["Início"] = df_input_a_trd["Início"].dt.time
df_input_a_trd["Fim"] = df_input_a_trd["Fim"].dt.time
df_input_a_trd['Nutri'] = df_input_a['Unnamed: 6'].copy()
df_input_a_trd.drop(columns=['Mês_num'], inplace=True)
df_input_a_trd = label_semana(df_input_a_trd)
df_input_a_trd['Mês'] = df_input_a_trd['Data'].dt.month
df_input_a_trd

,Data completa,Ano,Mês,Mês_Ano,DDS,Início,Fim,Total horas,Janelas,Nutri,Data,Semana_mes,Semana_label
0,05/01/2026 - Seg,2026,1,Janeiro2026,Seg,07:00:00,09:00:00,02:00:00,2,DANIELLE CASTELLANI,2026-01-05,2,2026 - 1 Jan - Sem 2 - 05 a 11
1,05/01/2026 - Seg,2026,1,Janeiro2026,Seg,09:00:00,12:00:00,03:00:00,3,MANOELA MARIOTTO LANZARA,2026-01-05,2,2026 - 1 Jan - Sem 2 - 05 a 11
2,05/01/2026 - Seg,2026,1,Janeiro2026,Seg,11:00:00,12:00:00,01:00:00,1,JULIANA RIBEIRO DE MELO,2026-01-05,2,2026 - 1 Jan - Sem 2 - 05 a 11
3,05/01/2026 - Seg,2026,1,Janeiro2026,Seg,12:00:00,13:00:00,01:00:00,1,JULIANA RIBEIRO DE MELO,2026-01-05,2,2026 - 1 Jan - Sem 2 - 05 a 11
4,05/01/2026 - Seg,2026,1,Janeiro2026,Seg,13:00:00,17:00:00,04:00:00,4,ANDREIA DUTRA GONCALVES,2026-01-05,2,2026 - 1 Jan - Sem 2 - 05 a 11
...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,30/01/2026 - Sex,2026,1,Janeiro2026,Sex,11:00:00,12:00:00,01:00:00,1,DANIELLE CASTELLANI,2026-01-30,5,2026 - 1 Jan - Sem 5 - 26 a 31
186,31/01/2026 - Sab,2026,1,Janeiro2026,Sab,08:00:00,09:00:00,01:00:00,1,JULIANA RIBEIRO DE MELO,2026-01-31,5,2026 - 1 Jan - Sem 5 - 26 a 31
187,31/01/2026 - Sab,2026,1,Janeiro2026,Sab,09:00:00,10:00:00,01:00:00,1,JULIANA RIBEIRO DE MELO,2026-01-31,5,2026 - 1 Jan - Sem 5 - 26 a 31
188,31/01/2026 - Sab,2026,1,Janeiro2026,Sab,10:00:00,11:00:00,01:00:00,1,JULIANA RIBEIRO DE MELO,2026-01-31,5,2026 - 1 Jan - Sem 5 - 26 a 31


In [10]:
#TODO EXCLUIR (ESTA ETAPA SERVE SOMENTE PARA TESTES)

df_input_a_trd = df_input_a_trd[df_input_a_trd['Semana_label'].str.contains('2026 - 1 Jan')]

## Input D - Análise de Agendamentos (Optum) Mensal

In [11]:
pasta_d = r'C:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\arquivos originais\Arquivos fechamento Jan_2026'
arquivos_d = glob.glob(f'{pasta_d}/relatorio-sessoes-realizadas*.xlsx')

cols_agendamentos = ['Número do caso', 'Beneficiário', 'Empresa', 'Data inicial do caso',
       'Status do caso', 'Tipo medida tomada', 'Responsável', 'Data sessão',
       'Tempo atendimento', 'Status sessão', 'Valor Unitário']

df_input_d = pd.concat([pd.read_excel(f, usecols=cols_agendamentos) for f in arquivos_d], ignore_index=True)
print(f'{len(arquivos_d)} arquivo(s) carregado(s)')
df_input_d.head()

9 arquivo(s) carregado(s)


c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook con

,Número do caso,Beneficiário,Empresa,Data inicial do caso,Status do caso,Tipo medida tomada,Responsável,Data sessão,Tempo atendimento,Status sessão,Valor Unitário
0,510418,Paula Ribeiro Nascimento,BOZEL BRASIL S A SODXP 447040,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,29/01/2026,00:45,Compareceu ao atendimento,"R$ 84,00"
1,510414,K tia C ssia Germana da Silva,BAT SOUZA CRUZ,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,29/01/2026,00:45,Faltou ao atendimento,"R$ 84,00"
2,510406,Giovanna Desideri de Paiva,IBM BRASIL INDUSTRIA MAQUINAS,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,29/01/2026,00:45,Faltou ao atendimento,"R$ 84,00"
3,510253,Cec lia Pereira Lopes,KOMATSU BRASIL INTERNATIONAL SODXP330343,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,29/01/2026,00:45,Compareceu ao atendimento,"R$ 84,00"
4,510070,Mylena Regina Silva de Lima,IBM BRASIL INDUSTRIA MAQUINAS,26/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,27/01/2026,00:45,Compareceu ao atendimento,"R$ 84,00"


In [12]:
df_input_d_trd = df_input_d.copy()
df_input_d_trd.rename(columns={'Data sessão':'Data', 'Responsável':'Nutri'}, inplace=True)
df_input_d_trd["Data"] = pd.to_datetime(df_input_d_trd["Data"], format="%d/%m/%Y", errors="coerce")
df_input_d_trd = label_semana(df_input_d_trd)
df_input_d_trd['Ano'] = df_input_d_trd['Data'].dt.year
df_input_d_trd['Mês'] = df_input_d_trd['Data'].dt.month
df_input_d_trd.head()

,Número do caso,Beneficiário,Empresa,Data inicial do caso,Status do caso,Tipo medida tomada,Nutri,Data,Tempo atendimento,Status sessão,Valor Unitário,Semana_mes,Semana_label,Ano,Mês
0,510418,Paula Ribeiro Nascimento,BOZEL BRASIL S A SODXP 447040,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,2026-01-29,00:45,Compareceu ao atendimento,"R$ 84,00",5,2026 - 1 Jan - Sem 5 - 26 a 31,2026,1
1,510414,K tia C ssia Germana da Silva,BAT SOUZA CRUZ,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,2026-01-29,00:45,Faltou ao atendimento,"R$ 84,00",5,2026 - 1 Jan - Sem 5 - 26 a 31,2026,1
2,510406,Giovanna Desideri de Paiva,IBM BRASIL INDUSTRIA MAQUINAS,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,2026-01-29,00:45,Faltou ao atendimento,"R$ 84,00",5,2026 - 1 Jan - Sem 5 - 26 a 31,2026,1
3,510253,Cec lia Pereira Lopes,KOMATSU BRASIL INTERNATIONAL SODXP330343,27/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,2026-01-29,00:45,Compareceu ao atendimento,"R$ 84,00",5,2026 - 1 Jan - Sem 5 - 26 a 31,2026,1
4,510070,Mylena Regina Silva de Lima,IBM BRASIL INDUSTRIA MAQUINAS,26/01/2026,Atualizar caso,Atendimento individual digital,ANA LU SA R SZAJUBOK,2026-01-27,00:45,Compareceu ao atendimento,"R$ 84,00",5,2026 - 1 Jan - Sem 5 - 26 a 31,2026,1


In [13]:
# Tratamento de nomes
df_input_d_trd = tratar_nomes_nutri(df_input_d_trd, coluna='Nutri')

In [14]:
#TODO EXCLUIR (ESTA ETAPA SERVE SOMENTE PARA TESTES)

df_input_d_trd = df_input_d_trd[df_input_d_trd['Semana_label'].str.contains('2026 - 1 Jan')]

## Output C - Disponibilidade x Agendamentos

In [15]:
df_output_c = df_input_a_trd.pivot_table(index='Semana_label',
                                         columns='Nutri',
                                         values='Janelas',
                                         aggfunc='sum',
                                         fill_value=0,
                                         dropna=False)
df_output_c['TOTAL'] = df_output_c.sum(axis=1)
df_output_c['CHECK'] = 'Oferta'

tb_temp = df_input_d_trd.pivot_table(index='Semana_label',
                                         columns='Nutri',
                                         values='Número do caso',
                                         aggfunc='count',
                                         fill_value=0,
                                         dropna=False)
tb_temp['TOTAL'] = tb_temp.sum(axis=1)
tb_temp['CHECK'] = 'Ocupação'

df_output_c = pd.concat([df_output_c, tb_temp])

del tb_temp

In [16]:
cols = list(df_output_c.columns)

# Identificar colunas especiais
col_check = ['CHECK']
col_total = ['TOTAL']

# Remover especiais da lista
middle_cols = [c for c in cols if c not in ['CHECK', 'TOTAL']]

# Ordenar as nutricionistas
middle_cols_sorted = sorted(middle_cols)

# Novo ordenamento final
new_order = col_check + middle_cols_sorted + col_total

df_output_c = df_output_c[new_order]
df_output_c = df_output_c.fillna(0)
df_output_c[middle_cols_sorted] = df_output_c[middle_cols_sorted].astype(int)
df_output_c = df_output_c.sort_index()
df_output_c

Nutri,CHECK,ANA LUÍSA R. SZAJUBOK,ANDREIA DUTRA GONCALVES,CÍNTIA DOS SANTOS IRINEU,DANIELLE CASTELLANI,ISADORA BEATRIZ ROSSI AMATO,JOSE RIBEIRO GOUVEIA NETO,JULIANA RIBEIRO DE MELO,MANOELA MARIOTTO LANZARA,STEPHANY PATRICIA PEREZ MARTINEZ,TOTAL
Semana_label,,,,,,,,,,,
2026 - 1 Jan - Sem 2 - 05 a 11,Oferta,16,16,0,16,0,10,16,10,16,100
2026 - 1 Jan - Sem 2 - 05 a 11,Ocupação,14,13,0,14,0,9,13,10,10,83
2026 - 1 Jan - Sem 3 - 12 a 18,Oferta,16,16,6,16,1,9,0,16,16,96
2026 - 1 Jan - Sem 3 - 12 a 18,Ocupação,9,15,1,15,1,7,0,12,11,71
2026 - 1 Jan - Sem 4 - 19 a 25,Oferta,16,16,15,16,1,10,23,10,16,123
2026 - 1 Jan - Sem 4 - 19 a 25,Ocupação,13,16,4,16,1,8,16,9,14,97
2026 - 1 Jan - Sem 5 - 26 a 31,Oferta,16,16,12,16,1,10,24,6,16,117
2026 - 1 Jan - Sem 5 - 26 a 31,Ocupação,13,12,9,15,1,9,19,6,14,98


In [17]:
# 1) Tabela resumida por semana, separando Oferta e Ocupação em colunas
df_semana = df_output_c.pivot_table(
    index="Semana_label",
    columns="CHECK",
    values="TOTAL",
    aggfunc="sum",
    fill_value=0
)

# Garante as colunas, mesmo que alguma semana não tenha Ocupação ainda
for col in ["Oferta", "Ocupação"]:
    if col not in df_semana.columns:
        df_semana[col] = 0

# 2) Calcula percentuais
df_semana["Agenda Ocupação"] = (
    df_semana["Ocupação"] / df_semana["Oferta"]
).replace([np.inf, np.nan], 0) * 100

df_semana["Horários Vagos"] = (
    100 - df_semana["Agenda Ocupação"]
)

# Arredondar
df_semana["Agenda Ocupação"] = df_semana["Agenda Ocupação"].round(2)
df_semana["Horários Vagos"] = df_semana["Horários Vagos"].round(2)

df_semana

CHECK,Ocupação,Oferta,Agenda Ocupação,Horários Vagos
Semana_label,,,,
2026 - 1 Jan - Sem 2 - 05 a 11,83,100,83.00,17.00
2026 - 1 Jan - Sem 3 - 12 a 18,71,96,73.96,26.04
2026 - 1 Jan - Sem 4 - 19 a 25,97,123,78.86,21.14
2026 - 1 Jan - Sem 5 - 26 a 31,98,117,83.76,16.24


In [18]:
# oferta por nutri
oferta_nutri = df_output_c[df_output_c["CHECK"] == "Oferta"].drop(columns=["CHECK"])

# ocupação por nutri
ocupacao_nutri = df_output_c[df_output_c["CHECK"] == "Ocupação"].drop(columns=["CHECK"])

# somatório total por coluna
oferta_total = oferta_nutri.sum()
ocupacao_total = ocupacao_nutri.sum()

percent_ocupacao = (ocupacao_total / oferta_total * 100).round(1)
percent_vago = (100 - percent_ocupacao).round(1)

df_percent_nutri = pd.DataFrame({
    "Oferta": oferta_total,
    "Ocupação": ocupacao_total,
    "% Ocupação": percent_ocupacao,
    "% Horários Vagos": percent_vago
})

df_percent_nutri

,Oferta,Ocupação,% Ocupação,% Horários Vagos
Nutri,,,,
ANA LUÍSA R. SZAJUBOK,64,49,76.6,23.4
ANDREIA DUTRA GONCALVES,64,56,87.5,12.5
CÍNTIA DOS SANTOS IRINEU,33,14,42.4,57.6
DANIELLE CASTELLANI,64,60,93.8,6.2
ISADORA BEATRIZ ROSSI AMATO,3,3,100.0,0.0
JOSE RIBEIRO GOUVEIA NETO,39,33,84.6,15.4
JULIANA RIBEIRO DE MELO,63,48,76.2,23.8
MANOELA MARIOTTO LANZARA,42,37,88.1,11.9
STEPHANY PATRICIA PEREZ MARTINEZ,64,49,76.6,23.4


## Input E - Análise de Atendimentos Realizados

In [19]:
pasta = r'C:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\arquivos originais\Arquivos fechamento Jan_2026'
colunas_necessarias = ['Data ', 'Nutri ', 'ID caso', 'Status atendimento \n(Realizado, Falta, Reagendou)']
arquivos = glob.glob(os.path.join(pasta, "*Controle de atendimentos*.xlsx"))

df_list = []

print("Iniciando leitura dos arquivos...\n")

#TODO É importante que o feedback esteja no dashboard, para que o usuário tenha perspectiva de que há ou não problemas nos dados sendo carregados
for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    
    try:
        # Lê apenas cabeçalho para validar colunas
        df_tmp = pd.read_excel(arquivo, skiprows=2, nrows=0, sheet_name='Controle atendimentos')
        colunas_encontradas = df_tmp.columns.tolist()

        # Verifica se todas as colunas obrigatórias existem
        if all(c in colunas_encontradas for c in colunas_necessarias):
            print(f"[OK] Arquivo aceito: {nome}")
            
            # Agora lê o arquivo completo (somente colunas necessárias)
            df = pd.read_excel(arquivo, skiprows=2, usecols=colunas_necessarias, sheet_name='Controle atendimentos')
            df["Arquivo"] = nome  # opcional: para rastrear a origem
            df_list.append(df)

        else:
            print(f"[ERRO] Arquivo rejeitado (faltam colunas): {nome}")
            print("→ Colunas encontradas:", colunas_encontradas)
            print("→ Colunas necessárias:", colunas_necessarias)
            print("-" * 60)

    except Exception as e:
        print(f"[FALHA] Não foi possível ler o arquivo: {nome}")
        print("Erro:", e)
        print("-" * 60)

print("\nFinalizando processamento...\n")

df_input_e = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
df_input_e.head()

Iniciando leitura dos arquivos...

[OK] Arquivo aceito: 02 Controle de atendimentos_AnaLu.xlsx
[OK] Arquivo aceito: 02 Controle de atendimentos_Andreia.xlsx


c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


[OK] Arquivo aceito: 02 Controle de atendimentos_Cintia.xlsx


c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


[OK] Arquivo aceito: 02 Controle de atendimentos_Danielle.xlsx
[OK] Arquivo aceito: 02 Controle de atendimentos_Isa.xlsx


...

[OK] Arquivo aceito: 02 Controle de atendimentos_Juliana.xlsx
[OK] Arquivo aceito: 02 Controle de atendimentos_Manoela.xlsx
[OK] Arquivo aceito: 02 Controle de atendimentos_Neto.xlsx


c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


[OK] Arquivo aceito: 02 Controle de atendimentos_Stephany.xlsx

Finalizando processamento...



c:\Users\vhcna\OneDrive - Marca Saúde\Área de Trabalho\Freelas\Flua nutrição\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Data,Nutri,ID caso,"Status atendimento \n(Realizado, Falta, Reagendou)",Arquivo
0,2024-02-27 00:00:00,ANA LU SA R SZAJUBOK,408222.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
1,2024-02-27 00:00:00,ANA LU SA R SZAJUBOK,408015.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
2,2024-02-27 00:00:00,ANA LU SA R SZAJUBOK,407911.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx
3,2024-02-29 00:00:00,ANA LU SA R SZAJUBOK,408854.0,Falta,02 Controle de atendimentos_AnaLu.xlsx
4,2024-03-05 00:00:00,ANA LU SA R SZAJUBOK,408052.0,Falta,02 Controle de atendimentos_AnaLu.xlsx


In [20]:
df_input_e_trd = df_input_e[~df_input_e['Data '].isnull()]
df_input_e_trd.rename(columns={'Data ':'Data', 'Nutri ':'Nutri'}, inplace=True)
df_input_e_trd["Data"] = pd.to_datetime(df_input_e_trd["Data"], format="%d/%m/%Y", errors="coerce")
df_input_e_trd = label_semana(df_input_e_trd)
df_input_e_trd['Ano'] = df_input_e_trd['Data'].dt.year
df_input_e_trd['Mês'] = df_input_e_trd['Data'].dt.month
df_input_e_trd = tratar_nomes_nutri(df_input_e_trd, coluna='Nutri') # Tratar nomes
# df_input_e_trd = df_input_e_trd[df_input_e_trd['Status atendimento \n(Realizado, Falta, Reagendou)'].isin(['Realizado','Realizado '])]
df_input_e_trd.head()

C:\Users\vhcna\AppData\Local\Temp\ipykernel_19964\177759609.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_input_e_trd.rename(columns={'Data ':'Data', 'Nutri ':'Nutri'}, inplace=True)
C:\Users\vhcna\AppData\Local\Temp\ipykernel_19964\177759609.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_input_e_trd["Data"] = pd.to_datetime(df_input_e_trd["Data"], format="%d/%m/%Y", errors="coerce")
C:\Users\vhcna\AppData\Local\Temp\ipykernel_19964\495743910.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

,Data,Nutri,ID caso,"Status atendimento \n(Realizado, Falta, Reagendou)",Arquivo,Semana_mes,Semana_label,Ano,Mês
0,2024-02-27,ANA LUÍSA R. SZAJUBOK,408222.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx,5,2024 - 2 Fev - Sem 5 - 26 a 29,2024.0,2.0
1,2024-02-27,ANA LUÍSA R. SZAJUBOK,408015.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx,5,2024 - 2 Fev - Sem 5 - 26 a 29,2024.0,2.0
2,2024-02-27,ANA LUÍSA R. SZAJUBOK,407911.0,Realizado,02 Controle de atendimentos_AnaLu.xlsx,5,2024 - 2 Fev - Sem 5 - 26 a 29,2024.0,2.0
3,2024-02-29,ANA LUÍSA R. SZAJUBOK,408854.0,Falta,02 Controle de atendimentos_AnaLu.xlsx,5,2024 - 2 Fev - Sem 5 - 26 a 29,2024.0,2.0
4,2024-03-05,ANA LUÍSA R. SZAJUBOK,408052.0,Falta,02 Controle de atendimentos_AnaLu.xlsx,2,2024 - 3 Mar - Sem 2 - 04 a 10,2024.0,3.0


In [21]:
#TODO EXCLUIR (CELULA PARA TESTE)

df_input_e_trd = df_input_e_trd[df_input_e_trd['Semana_label'].str.contains('2026 - 1 Jan')]

## Output F - Oferta x Agendamento x Atendimento

In [22]:
df_output_f = df_input_a_trd.groupby(['Nutri', 'Ano', 'Mês'], as_index=False, dropna=False)['Janelas'].sum()
df_output_f.rename(columns={'Janelas':'Oferta'}, inplace=True)

tb_temp = df_input_d_trd.groupby(['Nutri', 'Ano', 'Mês'], as_index=False, dropna=False)['Número do caso'].count()
tb_temp.rename(columns={'Número do caso':'Ocupação'}, inplace=True)

df_output_f = df_output_f.merge(tb_temp, on=['Nutri', 'Ano', 'Mês'], how='outer', validate='1:1')

tb_temp = df_input_e_trd.groupby(['Nutri', 'Ano', 'Mês'], as_index=False, dropna=False)['ID caso'].count()
tb_temp.rename(columns={'ID caso':'Realizado'}, inplace=True)

df_output_f = df_output_f.merge(tb_temp, on=['Nutri', 'Ano', 'Mês'], how='outer', validate='1:1')

df_output_f = df_output_f.fillna(0)
df_output_f["% Ocupação"] = (df_output_f["Ocupação"] / df_output_f["Oferta"]).replace([np.inf, np.nan], 0) * 100
df_output_f["% Realizado"] = (df_output_f["Realizado"] / df_output_f["Ocupação"]).replace([np.inf, np.nan], 0) * 100
df_output_f = df_output_f.sort_values(['Ano','Mês'])
df_output_f

,Nutri,Ano,Mês,Oferta,Ocupação,Realizado,% Ocupação,% Realizado
0,ANA LUÍSA R. SZAJUBOK,2026,1,64,49,49,76.562500,100.000000
1,ANDREIA DUTRA GONCALVES,2026,1,64,56,56,87.500000,100.000000
2,CÍNTIA DOS SANTOS IRINEU,2026,1,33,14,15,42.424242,107.142857
3,DANIELLE CASTELLANI,2026,1,64,60,60,93.750000,100.000000
4,ISADORA BEATRIZ ROSSI AMATO,2026,1,3,3,3,100.000000,100.000000
5,JOSE RIBEIRO GOUVEIA NETO,2026,1,39,33,33,84.615385,100.000000
6,JULIANA RIBEIRO DE MELO,2026,1,63,48,48,76.190476,100.000000
7,MANOELA MARIOTTO LANZARA,2026,1,42,37,37,88.095238,100.000000
8,STEPHANY PATRICIA PEREZ MARTINEZ,2026,1,64,49,49,76.562500,100.000000


In [23]:
cols = ['Ano', 'Mês', 'Oferta', 'Ocupação', 'Realizado']

df_output_f = df_output_f[~(df_output_f[cols] == 0).all(axis=1)]

check_one = df_output_f[(df_output_f[['Ano', 'Mês']] == 0).all(axis=1)&(df_output_f[['Oferta', 'Ocupação', 'Realizado']] > 0).any(axis=1)]

#TODO É importante que esse feedback esteja no dash, para o usuário ter perspectiva de que há problemas nos dados
if check_one.shape[0] > 0:
    print('Há eventos lançados com problema na informação de data')
    display(check_one)

df_output_f = df_output_f[~df_output_f.index.isin(check_one.index)]

In [24]:
df_graph_h = df_output_f.groupby(['Ano','Mês'], as_index=False, dropna=False)[['Oferta','Ocupação','Realizado']].sum()
df_graph_h["% Ocupação"] = (df_graph_h["Ocupação"] / df_graph_h["Oferta"]).replace([np.inf, np.nan], 0) * 100
df_graph_h["% Realizado"] = (df_graph_h["Realizado"] / df_graph_h["Ocupação"]).replace([np.inf, np.nan], 0) * 100
df_graph_h['Competencia'] = pd.to_datetime({"year": df_graph_h["Ano"], "month": df_graph_h["Mês"], "day": 1})

In [38]:
df_plot = df_graph_h.sort_values("Competencia").copy()
df_plot["Competencia"] = df_plot["Competencia"].dt.strftime("%b_%y")

grafico_barra_linha(df_plot, "Competencia")

In [26]:
#TODO aqui no código estou filtrando o mês, mas a ideia é que no dashboard isso seja escolhido pelo usuário
filtro_mes = 1

df_graph_i = df_input_a_trd[df_input_a_trd['Mês']==filtro_mes]
df_graph_i['Dia_semana_cod'] = df_graph_i['Data'].dt.weekday
df_graph_i['Dia_semana_desc'] = df_graph_i['Dia_semana_cod'].map(dict_dia_semana)
df_graph_i = df_graph_i.groupby(['Dia_semana_cod','Dia_semana_desc'], as_index=False, dropna=False)['Janelas'].sum()
df_graph_i.rename(columns={'Janelas':'Oferta'}, inplace=True)

tb_temp = df_input_d_trd[df_input_d_trd['Mês']==filtro_mes]
tb_temp['Dia_semana_cod'] = tb_temp['Data'].dt.weekday
tb_temp['Dia_semana_desc'] = tb_temp['Dia_semana_cod'].map(dict_dia_semana)
tb_temp = tb_temp.groupby(['Dia_semana_cod','Dia_semana_desc'], as_index=False, dropna=False)['Número do caso'].count()
tb_temp.rename(columns={'Número do caso':'Ocupação'}, inplace=True)

df_graph_i = df_graph_i.merge(tb_temp, on=['Dia_semana_cod','Dia_semana_desc'], how='outer', validate='1:1')

tb_temp = df_input_e_trd[df_input_e_trd['Mês']==filtro_mes]
tb_temp['Dia_semana_cod'] = tb_temp['Data'].dt.weekday
tb_temp['Dia_semana_desc'] = tb_temp['Dia_semana_cod'].map(dict_dia_semana)
tb_temp = tb_temp.groupby(['Dia_semana_cod','Dia_semana_desc'], as_index=False, dropna=False)['ID caso'].count()
tb_temp.rename(columns={'ID caso':'Realizado'}, inplace=True)

df_graph_i = df_graph_i.merge(tb_temp, on=['Dia_semana_cod','Dia_semana_desc'], how='outer', validate='1:1')

df_graph_i.drop(columns=['Dia_semana_cod'], inplace=True)
df_graph_i = df_graph_i.fillna(0)
df_graph_i["% Ocupação"] = (df_graph_i["Ocupação"] / df_graph_i["Oferta"]).replace([np.inf, np.nan], 0) * 100
df_graph_i["% Realizado"] = (df_graph_i["Realizado"] / df_graph_i["Ocupação"]).replace([np.inf, np.nan], 0) * 100

df_graph_i

,Dia_semana_desc,Oferta,Ocupação,Realizado,% Ocupação,% Realizado
0,Seg,91,72,72,79.120879,100.000000
1,Ter,133,90,90,67.669173,100.000000
2,Qua,86,71,71,82.558140,100.000000
3,Qui,74,65,66,87.837838,101.538462
4,Sex,43,42,42,97.674419,100.000000
5,Sáb,9,9,9,100.000000,100.000000


In [27]:
grafico_barra_linha(df_graph_i, "Dia_semana_desc")

In [28]:
#TODO deslocar o eixo secundário para as linhas ficarem acima das barras

## Output G

In [29]:
df_output_g = df_input_d_trd.copy()

df_output_g['Valor Unitário'] = (
    df_output_g["Valor Unitário"]
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .astype(float)
)

df_output_g = df_output_g.pivot_table(index=['Mês','Nutri'],
                                      columns='Status sessão',
                                      aggfunc={'Número do caso':'count', 'Valor Unitário':'sum'},
                                      fill_value=0,
                                      dropna=False)

df_output_g['Total Número do caso'] = df_output_g['Número do caso'].sum(axis=1)
df_output_g['Total Valor Unitário'] = df_output_g['Valor Unitário'].sum(axis=1)

In [30]:
tb_temp = df_input_e_trd.groupby(['Mês','Nutri'], dropna=False, as_index=False)[['ID caso']].count()
tb_temp.rename(columns={'ID caso': 'Casos (Planilhas)'}, inplace=True)
df_left = df_output_g[['Total Número do caso']].reset_index()
df_left.columns = ['Mês', 'Nutri', 'Total Número do caso']
tb_temp = df_left.merge(tb_temp, on=['Mês','Nutri'], how='left', validate='1:1')
tb_temp['Check'] = tb_temp['Total Número do caso'] == tb_temp['Casos (Planilhas)']
tb_temp['Check'] = tb_temp['Check'].map({False:'Not OK', True:'OK'})
tb_temp = tb_temp.set_index(['Mês','Nutri'])

dict_check = tb_temp['Check'].to_dict()
dict_optum = tb_temp['Casos (Planilhas)'].to_dict()

In [31]:
df_output_g['Casos (Planilhas)'] = df_output_g.index.map(dict_optum)
df_output_g['Check'] = df_output_g.index.map(dict_check)
df_output_g

Número do caso  \
Status sessão                        Compareceu ao atendimento   
Mês Nutri                                                        
1   ANA LUÍSA R. SZAJUBOK                                   29   
    ANDREIA DUTRA GONCALVES                                 29   
    CÍNTIA DOS SANTOS IRINEU                                10   
    DANIELLE CASTELLANI                                     38   
    ISADORA BEATRIZ ROSSI AMATO                              1   
    JOSE RIBEIRO GOUVEIA NETO                               21   
    JULIANA RIBEIRO DE MELO                                 27   
    MANOELA MARIOTTO LANZARA                                23   
    STEPHANY PATRICIA PEREZ MARTINEZ                        29   

                                                            \
Status sessão                        Faltou ao atendimento   
Mês Nutri                                                    
1   ANA LUÍSA R. SZAJUBOK                               20   
    ANDREIA DUTRA GONCALVES                             27   
    CÍNTIA DOS SANTOS IRINEU                             4   
    DANIELLE CASTELLANI                                 22   
    ISADORA BEATRIZ ROSSI AMATO                          2   
    JOSE RIBEIRO GOUVEIA NETO                           12   
    JULIANA RIBEIRO DE MELO                             21   
    MANOELA MARIOTTO LANZARA                            14   
    STEPHANY PATRICIA PEREZ MARTINEZ                    20   

                                                Valor Unitário  \
Status sessão                        Compareceu ao atendimento   
Mês Nutri                                                        
1   ANA LUÍSA R. SZAJUBOK                               2436.0   
    ANDREIA DUTRA GONCALVES                             2436.0   
    CÍNTIA DOS SANTOS IRINEU                             840.0   
    DANIELLE CASTELLANI                                 3192.0   
    ISADORA BEATRIZ ROSSI AMATO                           84.0   
    JOSE RIBEIRO GOUVEIA NETO                           1764.0   
    JULIANA RIBEIRO DE MELO                             2268.0   
    MANOELA MARIOTTO LANZARA                            1932.0   
    STEPHANY PATRICIA PEREZ MARTINEZ                    2436.0   

                                                            \
Status sessão                        Faltou ao atendimento   
Mês Nutri                                                    
1   ANA LUÍSA R. SZAJUBOK                           1680.0   
    ANDREIA DUTRA GONCALVES                         2268.0   
    CÍNTIA DOS SANTOS IRINEU                         336.0   
    DANIELLE CASTELLANI                             1848.0   
    ISADORA BEATRIZ ROSSI AMATO                      168.0   
    JOSE RIBEIRO GOUVEIA NETO                       1008.0   
    JULIANA RIBEIRO DE MELO                         1764.0   
    MANOELA MARIOTTO LANZARA                        1176.0   
    STEPHANY PATRICIA PEREZ MARTINEZ                1680.0   

                                     Total Número do caso  \
Status sessão                                               
Mês Nutri                                                   
1   ANA LUÍSA R. SZAJUBOK                              49   
    ANDREIA DUTRA GONCALVES                            56   
    CÍNTIA DOS SANTOS IRINEU                           14   
    DANIELLE CASTELLANI                                60   
    ISADORA BEATRIZ ROSSI AMATO                         3   
    JOSE RIBEIRO GOUVEIA NETO                          33   
    JULIANA RIBEIRO DE MELO                            48   
    MANOELA MARIOTTO LANZARA                           37   
    STEPHANY PATRICIA PEREZ MARTINEZ                   49   

                                     Total Valor Unitário Casos (Planilhas)  \
Status sessão                                                                 
Mês Nutri                                                                     
1 

In [32]:
#TODO comparar os ID do paciente entre Optum e planilhas das Nutris

In [33]:
df_output_h = df_input_e_trd.groupby(['Ano','Mês','Nutri','ID caso'], as_index=False, dropna=False)['Data'].count()
df_output_h['Ano'] = df_output_h['Ano'].astype(int)
df_output_h['Mês'] = df_output_h['Mês'].astype(int)
df_output_h['ID caso'] = df_output_h['ID caso'].astype(int)
df_output_h = df_output_h.rename(columns={'Data':'Planilhas'})

tb_temp = df_input_d_trd.groupby(['Ano','Mês','Nutri','Número do caso'], as_index=False, dropna=False)['Data'].count()
tb_temp = tb_temp.rename(columns={'Número do caso':'ID caso', 'Data':'Optum'})

df_output_h = df_output_h.merge(tb_temp, on=['Ano','Mês','Nutri','ID caso'], how='outer', validate='1:1')
df_output_h['Check'] = df_output_h['Planilhas'] == df_output_h['Optum']
df_output_h = df_output_h[df_output_h['Check']==False]
df_output_h

,Ano,Mês,Nutri,ID caso,Planilhas,Optum,Check
98,2026,1,CÍNTIA DOS SANTOS IRINEU,509544,2,1,False


In [34]:
#TODO criar filtros de mês e nutri no streamlit

# Validações

### Validação Oferta e Ocupação Dez/25

In [ ]:
df_output_f

,Nutri,Ano,Mês,Oferta,Ocupação,Realizado,% Ocupação,% Realizado
0,ANA LUÍSA R. SZAJUBOK,2026,1,64,49,49,76.562500,100.000000
1,ANDREIA DUTRA GONCALVES,2026,1,64,56,50,87.500000,89.285714
2,CÍNTIA DOS SANTOS IRINEU,2026,1,33,14,15,42.424242,107.142857
3,DANIELLE CASTELLANI,2026,1,64,60,60,93.750000,100.000000
4,ISADORA BEATRIZ ROSSI AMATO,2026,1,3,3,3,100.000000,100.000000
5,JOSE RIBEIRO GOUVEIA NETO,2026,1,39,33,33,84.615385,100.000000
6,JULIANA RIBEIRO DE MELO,2026,1,63,48,44,76.190476,91.666667
7,MANOELA MARIOTTO LANZARA,2026,1,42,37,37,88.095238,100.000000
8,STEPHANY PATRICIA PEREZ MARTINEZ,2026,1,64,49,49,76.562500,100.000000


In [ ]:
df_validacao_oferta = pd.DataFrame(
             data={
                 'Nutri':['ALINE BATISTA RIBEIRO', 'ANA LUÍSA R. SZAJUBOK', 'ANDREIA DUTRA GONCALVES', 'CÍNTIA DOS SANTOS IRINEU', 'DANIELLE CASTELLANI', 'ISADORA BEATRIZ ROSSI AMATO', 'JOSE RIBEIRO GOUVEIA NETO', 'JULIANA RIBEIRO DE MELO', 'MANOELA MARIOTTO LANZARA', 'STEPHANY PATRICIA PEREZ MARTINEZ', 'TOTAL'],
                 'real_oferta':[36, 55, 49, 36, 48, 36, 36, 64, 33, 48, 441]
             })

df_validacao_oferta = df_validacao_oferta.merge(df_output_f[['Nutri','Oferta']], on='Nutri', how='outer', validate='1:1')
df_validacao_oferta.loc[df_validacao_oferta['Nutri']=='TOTAL', 'Oferta'] = df_validacao_oferta['Oferta'].sum()
df_validacao_oferta['validacao_oferta'] = df_validacao_oferta['real_oferta'] - df_validacao_oferta['Oferta']
df_validacao_oferta

,Nutri,real_oferta,Oferta,validacao_oferta
0,ALINE BATISTA RIBEIRO,36,NaN,NaN
1,ANA LUÍSA R. SZAJUBOK,55,64.0,-9.0
2,ANDREIA DUTRA GONCALVES,49,64.0,-15.0
3,CÍNTIA DOS SANTOS IRINEU,36,33.0,3.0
4,DANIELLE CASTELLANI,48,64.0,-16.0
5,ISADORA BEATRIZ ROSSI AMATO,36,3.0,33.0
6,JOSE RIBEIRO GOUVEIA NETO,36,39.0,-3.0
7,JULIANA RIBEIRO DE MELO,64,63.0,1.0
8,MANOELA MARIOTTO LANZARA,33,42.0,-9.0
9,STEPHANY PATRICIA PEREZ MARTINEZ,48,64.0,-16.0


In [ ]:
df_validacao_ocupacao = pd.DataFrame(
             data={
                 'Nutri':['ALINE BATISTA RIBEIRO', 'ANA LUÍSA R. SZAJUBOK', 'ANDREIA DUTRA GONCALVES', 'CÍNTIA DOS SANTOS IRINEU', 'DANIELLE CASTELLANI', 'ISADORA BEATRIZ ROSSI AMATO', 'JOSE RIBEIRO GOUVEIA NETO', 'JULIANA RIBEIRO DE MELO', 'MANOELA MARIOTTO LANZARA', 'STEPHANY PATRICIA PEREZ MARTINEZ', 'TOTAL'],
                 'real_ocupacao': [9, 34, 42, 26, 38, 18, 18, 49, 28, 20, 282]
             })

df_validacao_ocupacao = df_validacao_ocupacao.merge(df_output_f[['Nutri','Ocupação']], on='Nutri', how='outer', validate='1:1')
df_validacao_ocupacao.loc[df_validacao_ocupacao['Nutri']=='TOTAL', 'Ocupação'] = df_validacao_ocupacao['Ocupação'].sum()
df_validacao_ocupacao['validacao_ocupacao'] = df_validacao_ocupacao['real_ocupacao'] - df_validacao_ocupacao['Ocupação']
df_validacao_ocupacao

,Nutri,real_ocupacao,Ocupação,validacao_ocupacao
0,ALINE BATISTA RIBEIRO,9,NaN,NaN
1,ANA LUÍSA R. SZAJUBOK,34,49.0,-15.0
2,ANDREIA DUTRA GONCALVES,42,56.0,-14.0
3,CÍNTIA DOS SANTOS IRINEU,26,14.0,12.0
4,DANIELLE CASTELLANI,38,60.0,-22.0
5,ISADORA BEATRIZ ROSSI AMATO,18,3.0,15.0
6,JOSE RIBEIRO GOUVEIA NETO,18,33.0,-15.0
7,JULIANA RIBEIRO DE MELO,49,48.0,1.0
8,MANOELA MARIOTTO LANZARA,28,37.0,-9.0
9,STEPHANY PATRICIA PEREZ MARTINEZ,20,49.0,-29.0


In [ ]:
df_validacao_realizado = pd.DataFrame(
             data={
                 'Nutri':['ALINE BATISTA RIBEIRO', 'ANA LUÍSA R. SZAJUBOK', 'ANDREIA DUTRA GONCALVES', 'CÍNTIA DOS SANTOS IRINEU', 'DANIELLE CASTELLANI', 'ISADORA BEATRIZ ROSSI AMATO', 'JOSE RIBEIRO GOUVEIA NETO', 'JULIANA RIBEIRO DE MELO', 'MANOELA MARIOTTO LANZARA', 'STEPHANY PATRICIA PEREZ MARTINEZ', 'TOTAL'],
                 'real_realizado': [2, 21, 27, 12, 18, 11, 14, 24, 18, 13, 160]
             })

df_validacao_realizado = df_validacao_realizado.merge(df_input_e_trd.loc[df_input_e_trd['Status atendimento \n(Realizado, Falta, Reagendou)'].isin(['Realizado','Realizado ']), ['Nutri','Realizado']], on='Nutri', how='outer', validate='1:1')
df_validacao_realizado.loc[df_validacao_realizado['Nutri']=='TOTAL', 'Realizado'] = df_validacao_realizado['Realizado'].sum()
df_validacao_realizado['validacao_realizado'] = df_validacao_realizado['real_realizado'] - df_validacao_realizado['Realizado']
df_validacao_realizado

KeyError: "['Realizado'] not in index"